# Fazit

Dieses Skript ist bereits eine sehr gute Grundlage für ein professionelles juristisches RAG-System.

Besonders stark sind:

    . die hierarchische Strukturierung,

    . das granulare Chunking,

    . und das LLM-basierte Enrichment.

Das entspricht architektonisch bereits vielen modernen Legal-AI-Systemen.

## Mögliche Verbesserungen: 
Cross-Reference Graph

Verlinkung zwischen Paragraphen.

# Dokumentation: Juristische RAG-Ingestion-Pipeline

Diese Pipeline dient der granularen und strukturierten Verarbeitung von deutschen Gesetzestexten (Fokus: Mietrecht aus dem BGB). Sie bricht komplexe Gesetzestexte bis auf die Ebene von Paragraphen, Absätzen und Nummern herunter, reichert sie mittels eines Large Language Models (LLM) mit Metadaten und Erklärungen an und speichert das Ergebnis in einer Chroma-Vektordatenbank für Retrieval-Augmented Generation (RAG).

---

## 🛠️ Systemarchitektur & Workflow

Der Datenfluss ist streng hierarchisch aufgebaut, um die feingliedrige Struktur des deutschen Rechtssystems beizubehalten:

1. **Dateiebene:** Einlesen von `.txt`-Dateien aus dem Verzeichnis `dokumente_mietrecht`.
2. **Paragraphenebene (`§`):** Trennung der Texte nach Paragraphennummern und Titeln mittels Regex.
3. **Absatzebene (`(1), (2)`):** Unterteilung der Paragraphen in ihre jeweiligen Absätze.
4. **Nummernebene (`1., 2.`):** Optionale tiefergehende Aufspaltung von Listen innerhalb eines Absatzes (inklusive Isolierung des Einleitungstextes).
5. **LLM-Enrichment:** Jedes granulare Textstück (Chunk) wird an `gpt-4o-mini` gesendet, um eine strukturierte Zusammenfassung und Keywords zu generieren.
6. **Vektorspeicherung:** Speicherung der finalen Dokumente (Inhalt + strukturierte Metadaten) in `Chroma`.

---
```text
Architekturübersicht:
Der Code besteht aus folgenden Hauptteilen:

Konfiguration
Embeddings & Vektor-Datenbank
LLM für juristische Anreicherung
Parser für Gesetzesstruktur
Referenz-Extraktion
LLM-Enrichment
Aufbau strukturierter Chunks
Speicherung in ChromaDB

1. Konfiguration

DOCUMENTS_DIR = "dokumente_mietrecht"
PERSIST_DIRECTORY = "chroma_legal_rag"

2. Embeddings & Vektor-Datenbank
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

3. LLM & Structured Output

llm_enrichment = ChatOpenAI(model="gpt-5.4-mini", temperature=0.1)
class EnrichmentOutput(BaseModel):


r"(?m)^\s*§\s*(\d+[a-zA-Z]*)[ \t]+([A-ZÄÖÜ][^\n]*)$"

BGB
 └── §535 Mietvertrag
      ├── Absatz 1
      │     ├── Nummer 1
      │     └── Nummer 2
      └── Absatz 2

    metadata = {
    "gesetz": "BGB",
    "paragraph": "535",
    "absatz": "1",
    "nummer": "2",
    "source": "mietrecht.txt"
}

## 📂 Verzeichnisstruktur

Das Skript erwartet und erstellt die folgenden Ordnerstrukturen im lokalen Verzeichnis:


├── dein_skript.py             # Der Ingestion-Code
├── .env                       # Enthält den OpenAI API Key
├── dokumente_mietrecht/       # Eingabeordner für Rohtexte (.txt)
│   └── Mietrecht_kuendigung.txt, Mietrecht_mietzahlung.txt
└── chroma_legal_rag/          # Von Chroma automatisch generierte DB-Dateien

Beispiel von Chunk:
GESETZ: BGB
PARAGRAPH: §573
PARAGRAPH_TITEL: Ordentliche Kündigung des Vermieters
ABSATZ: 1
NUMMER: -
REFERENZEN: -
THEMA: Mietrecht
KEYWORDS: Vermieter, Kündigung, Mietverhältnis, berechtigtes Interesse, Mieterhöhung
TYPISCHE NUTZERFRAGEN:
- Wann kann ein Vermieter kündigen?
- Was ist ein berechtigtes Interesse?
- Kann ein Vermieter wegen Mieterhöhung kündigen?
KLARTEXT: Ein Vermieter darf einen Mietvertrag nur kündigen, wenn er einen guten Grund dafür hat. Eine Kündigung nur wegen einer geplanten Mieterhöhung ist nicht erlaubt.
ORIGINALTEXT: (1) Der Vermieter kann nur kündigen, wenn er ein berechtigtes Interesse an der Beendigung des Mietverhältnisses hat. Die Kündigung zum Zwecke der Mieterhöhung ist ausgeschlossen.

Zusammenfassung

| Komponente       | Aufgabe                            |
| ---------------- | ---------------------------------- |
| Regex-Parser     | Gesetzesstruktur erkennen          |
| Chunking         | Hierarchische Zerlegung            |
| LLM-Enrichment   | Juristische Analyse                |
| Embeddings       | Semantische Vektoren               |
| ChromaDB         | Persistente Speicherung            |
| Metadata         | Präzise Filterung                  |
| RAG-Vorbereitung | Grundlage für juristischen Chatbot |


# datei_1_ingestion.py

In [2]:
import os
import re
from typing import List, Optional, Dict
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

# ============================================================
# CONFIG
# ============================================================
load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY fehlt.")

DOCUMENTS_DIR = "dokumente_mietrecht"
PERSIST_DIRECTORY = "chroma_legal_rag"
os.makedirs(DOCUMENTS_DIR, exist_ok=True)

# ============================================================
# EMBEDDINGS & VEKTOR-DATENBANK
# ============================================================
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    persist_directory=PERSIST_DIRECTORY,
    embedding_function=embeddings
)

# ============================================================
# LLM & STRUCTURED OUTPUT
# llm_query = ChatOpenAI(model="gpt-5.4-nano", temperature=0.1)
# llm_answer = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
# ============================================================
llm_enrichment = ChatOpenAI(model="gpt-5.4-mini", temperature=0.1)

class EnrichmentOutput(BaseModel):
    topic: str = Field(description="Juristisches Hauptthema")
    plain_language_summary: str = Field(description="Einfache Erklärung")
    user_questions: List[str] = Field(description="Typische Nutzerfragen")
    keywords: List[str] = Field(description="Juristische Keywords")

structured_llm = llm_enrichment.with_structured_output(EnrichmentOutput)

# ============================================================
# HILFSFUNKTIONEN
# ============================================================
def normalize_text(text: str) -> str:
    """ Bereinigt typische OCR- und PDF-Leerzeichenprobleme """
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\u00A0", " ").replace("\u3000", " ")
    return text

# ============================================================
# JURISTISCHE STRUKTUR-PARSER
# ============================================================

def split_paragraphs(full_text: str) -> List[Dict]:
    full_text = normalize_text(full_text)
    
    # Greift die Kopfzeile sauber ab und lässt den Inhalt unangetastet
    paragraph_pattern = re.compile(r"(?m)^\s*§\s*(\d+[a-zA-Z]*)[ \t]+([A-ZÄÖÜ][^\n]*)$")
    #paragraph_pattern = re.compile(r"(?m)^\s*§\s*(\d+[a-zA-Z]*)(?:[ \t]+([^\n]+))?$")
    matches = list(paragraph_pattern.finditer(full_text))
    results = []

    if not matches:
        return [{
            "paragraph": "Unbekannt",
            "title": "Gesetzestext",
            "content": full_text.strip()
        }]

    for i, match in enumerate(matches):
        paragraph_number = str(match.group(1)).strip()
        paragraph_title = match.group(2)

        if paragraph_title and paragraph_title.strip().startswith("("):
            paragraph_title = ""
            start_content = match.end(1)
        else:
            paragraph_title = paragraph_title.strip() if paragraph_title else ""
            start_content = match.end()

        if i + 1 < len(matches):
            end_content = matches[i + 1].start()
        else:
            end_content = len(full_text)

        paragraph_text = full_text[start_content:end_content].strip()

        results.append({
            "paragraph": paragraph_number,
            "title": paragraph_title,
            "content": paragraph_text
        })

    return results


def split_absaetze(paragraph_text: str) -> List[Dict]:
    section_pattern = re.compile(r"(?m)^\s*\((\d+)\)")
    matches = list(section_pattern.finditer(paragraph_text))

    if not matches:
        return [{
            "absatz": "1", 
            "content": paragraph_text.strip()
        }]

    absaetze = []
    for i, match in enumerate(matches):
        absatz_number = str(match.group(1)).strip()
        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(paragraph_text)

        content = paragraph_text[start:end].strip()

        absaetze.append({
            "absatz": absatz_number,
            "content": content
        })

    return absaetze


def split_nummern(absatz_text: str) -> List[Dict]:
    nummer_pattern = re.compile(
        r"(\d+)\.(?!\s+(?:Januar|Februar|März|April|Mai|Juni|Juli|August|September|Oktober|November|Dezember))\s*"
    )
    
    matches = list(nummer_pattern.finditer(absatz_text))
    if not matches:
        return []

    intro_text = absatz_text[:matches[0].start()].strip()
    results = []

    for i, match in enumerate(matches):
        nummer = str(match.group(1)).strip()
        start = match.start()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(absatz_text)

        nummer_text = absatz_text[start:end].strip()
        full_content = f"{intro_text}\n{nummer_text}" if intro_text else nummer_text

        results.append({
            "nummer": nummer,
            "content": full_content,
            "intro_isolated": intro_text
        })

    return results


def extract_references(text: str) -> List[str]:
    refs = re.findall(r"§+\s*(\d+[a-zA-Z]*)", text)
    return list(set([str(r).strip() for r in refs]))

# ============================================================
# LLM ENRICHMENT
# ============================================================
def enrich_chunk(text: str) -> EnrichmentOutput:
    prompt = f"""Du bist ein juristischer KI-Assistent.
Analysiere den folgenden Gesetzestext.
Erzeuge ein strukturiertes JSON mit den geforderten Feldern.

Gesetzestext:
{text}"""

    try:
        return structured_llm.invoke(prompt)
    except Exception as e:
        print(f"  ⚠ LLM-Enrichment fehlgeschlagen: {e}")

    return EnrichmentOutput(
        topic="Unbekannt",
        plain_language_summary=text,
        user_questions=["Keine Fragen verfügbar."],
        keywords=[]
    )

# ============================================================
# CONTEXT STRUCTURING
# ============================================================
def build_page_content(
    gesetz: str, paragraph: str, paragraph_title: str, absatz: str,
    nummer: Optional[str], references: List[str], enrichment: EnrichmentOutput, original_text: str
) -> str:
    nummer_text = nummer if nummer else "-"
    references_text = ", ".join(references) if references else "-"
    user_questions = "\n".join([f"- {q}" for q in enrichment.user_questions])
    keywords = ", ".join(enrichment.keywords)

    return f"""GESETZ: {gesetz}
PARAGRAPH: §{paragraph}
PARAGRAPH_TITEL: {paragraph_title}
ABSATZ: {absatz}
NUMMER: {nummer_text}
REFERENZEN: {references_text}
THEMA: {enrichment.topic}
KEYWORDS: {keywords}
TYPISCHE NUTZERFRAGEN:
{user_questions}
KLARTEXT: {enrichment.plain_language_summary}
ORIGINALTEXT: {original_text}"""

# ============================================================
# INGESTION PIPELINE EXECUTION
# ============================================================
all_documents = []

for filename in os.listdir(DOCUMENTS_DIR):
    if not filename.endswith(".txt"):
        continue

    filepath = os.path.join(DOCUMENTS_DIR, filename)
    print(f"\n📄 Verarbeite Datei: {filename}")

    with open(filepath, "r", encoding="utf-8") as f:
        full_text = f.read()

    gesetz_name = "BGB"
    paragraphs = split_paragraphs(full_text)

    for para in paragraphs:
        paragraph_number = para["paragraph"]
        paragraph_title = para["title"]
        paragraph_content = para["content"]

        print(f"  ➡ Analysiere §{paragraph_number} {paragraph_title}")
        absaetze = split_absaetze(paragraph_content)

        for absatz_item in absaetze:
            absatz_number = absatz_item["absatz"]
            absatz_content = absatz_item["content"]

            nummern = split_nummern(absatz_content)

            if not nummern:
                original_text = absatz_content
                references = extract_references(original_text)
                enrichment = enrich_chunk(original_text)

                page_content = build_page_content(
                    gesetz=gesetz_name, paragraph=paragraph_number, paragraph_title=paragraph_title,
                    absatz=absatz_number, nummer=None, references=references, enrichment=enrichment, original_text=original_text
                )
                metadata = {
                    "gesetz": gesetz_name, "paragraph": str(paragraph_number),
                    "absatz": str(absatz_number), "nummer": "none", "source": filename
                }
                all_documents.append(Document(page_content=page_content, metadata=metadata))

            else:
                intro_text = nummern[0]["intro_isolated"]
                if intro_text and len(intro_text) > 10:
                    references = extract_references(intro_text)
                    enrichment = enrich_chunk(intro_text)

                    page_content = build_page_content(
                        gesetz=gesetz_name, paragraph=paragraph_number, paragraph_title=paragraph_title,
                        absatz=absatz_number, nummer="Einleitung", references=references, enrichment=enrichment, original_text=intro_text
                    )
                    metadata = {
                        "gesetz": gesetz_name, "paragraph": str(paragraph_number),
                        "absatz": str(absatz_number), "nummer": "intro", "source": filename
                    }
                    all_documents.append(Document(page_content=page_content, metadata=metadata))

                for nummer_item in nummern:
                    nummer = nummer_item["nummer"]
                    original_text = nummer_item["content"]
                    references = extract_references(original_text)
                    enrichment = enrich_chunk(original_text)

                    page_content = build_page_content(
                        gesetz=gesetz_name, paragraph=paragraph_number, paragraph_title=paragraph_title,
                        absatz=absatz_number, nummer=nummer, references=references, enrichment=enrichment, original_text=original_text
                    )
                    metadata = {
                        "gesetz": gesetz_name, "paragraph": str(paragraph_number),
                        "absatz": str(absatz_number), "nummer": str(nummer), "source": filename
                    }
                    all_documents.append(Document(page_content=page_content, metadata=metadata))

if all_documents:
    print(f"\n💾 Speichere {len(all_documents)} granulare Dokument-Chunks in Chroma...")
    vector_store.add_documents(all_documents)
    print("✅ Ingestion erfolgreich abgeschlossen.")


📄 Verarbeite Datei: Mietrecht_kuendigung.txt
  ➡ Analysiere §573 Ordentliche Kündigung des Vermieters
  ➡ Analysiere §573a Erleichterte Kündigung des Vermieters
  ➡ Analysiere §573c Fristen der ordentlichen Kündigung
  ➡ Analysiere §573d Außerordentliche Kündigung mit gesetzlicher Frist

📄 Verarbeite Datei: Mietrecht_mietzahlung.txt
  ➡ Analysiere §556 Vereinbarungen über Betriebskosten
  ➡ Analysiere §556a Abrechnungsmaßstab für Betriebskosten
  ➡ Analysiere §556b Fälligkeit der Miete, Aufrechnungs- und Zurückbehaltungsrecht
  ➡ Analysiere §556c Kosten der Wärmelieferung als Betriebskosten, Verordnungsermächtigung
  ➡ Analysiere §556d Zulässige Miethöhe bei Mietbeginn; Verordnungsermächtigung
  ➡ Analysiere §556e Berücksichtigung der Vormiete oder einer durchgeführten Modernisierung
  ➡ Analysiere §556f Ausnahmen
  ➡ Analysiere §556g Rechtsfolgen; Auskunft über die Miete

💾 Speichere 54 granulare Dokument-Chunks in Chroma...
✅ Ingestion erfolgreich abgeschlossen.


In [3]:
# Chroma-Datenbank laden (falls nicht schon geschehen)
# vector_store = Chroma(persist_directory=PERSIST_DIRECTORY, embedding_function=embeddings)

# Die ersten 5 Einträge abfragen
db_content = vector_store.get(limit=5)

print(f"Insgesamt in der DB gefunden: {len(db_content['documents'])} Chunks (zeige die ersten 5):\n")

for i in range(len(db_content['documents'])):
    print(f"=== CHUNK {i+1} ===")
    # Metadaten anzeigen (Gesetz, Paragraph, Absatz, Quelle)
    print(f"METADATEN: {db_content['metadatas'][i]}")
    print("-" * 40)
    # Den formatierten Textinhalt anzeigen
    print(db_content['documents'][i])
    print("=" * 40 + "\n")

Insgesamt in der DB gefunden: 5 Chunks (zeige die ersten 5):

=== CHUNK 1 ===
METADATEN: {'paragraph': '573', 'source': 'Mietrecht_kuendigung.txt', 'absatz': '1', 'nummer': 'none', 'gesetz': 'BGB'}
----------------------------------------
GESETZ: BGB
PARAGRAPH: §573
PARAGRAPH_TITEL: Ordentliche Kündigung des Vermieters
ABSATZ: 1
NUMMER: -
REFERENZEN: -
THEMA: Mietrecht
KEYWORDS: Vermieter, Kündigung, Mietverhältnis, berechtigtes Interesse, Mieterhöhung
TYPISCHE NUTZERFRAGEN:
- Wann kann ein Vermieter kündigen?
- Was ist ein berechtigtes Interesse?
- Kann ein Vermieter wegen Mieterhöhung kündigen?
KLARTEXT: Ein Vermieter darf einen Mietvertrag nur kündigen, wenn er einen guten Grund dafür hat. Eine Kündigung nur wegen einer geplanten Mieterhöhung ist nicht erlaubt.
ORIGINALTEXT: (1) Der Vermieter kann nur kündigen, wenn er ein berechtigtes Interesse an der Beendigung des Mietverhältnisses hat. Die Kündigung zum Zwecke der Mieterhöhung ist ausgeschlossen.

=== CHUNK 2 ===
METADATEN: {'nu

# datei_2_rag.py

# Dokumentation: Juristisches RAG-Anfragesystem (datei_2_rag.py)

Dieses Skript implementiert die **Retrieval- und Antwort-Komponente (Generation)** des juristischen RAG-Systems (Mietrecht). Es liest die zuvor in Chroma gespeicherten Dokumenten-Chunks ein, analysiert Nutzeranfragen mittels künstlicher Intelligenz auf Paragraphen-Nennungen, filtert die Vektordatenbank gezielt und generiert eine rechtssichere Antwort basierend auf dem gefundenen Kontext.

---

## 🛠️ Systemarchitektur & Ablaufkette (Chain)

Die Pipeline nutzt die LangChain Expression Language (LCEL) und folgt einem dreistufigen Prozess:

```text
 Nutzerfrage ──> [ 1. Query-Extraktion & Routing ]
                         │
                         ▼
                 [ 2. Retrieval Stage ] ──(Keine Treffer?)──> [ Fallback: Ungefilterte Suche ]
                         │
                         ▼
                 [ 3. Generation Stage ] ──> Rechtssichere Antwort


Nutzerfrage
    ↓
Query-Analyse (LLM)
    ↓
Metadata-Filter + Semantic Retrieval
    ↓
Fallback-Retrieval
    ↓
Kontextaufbau
    ↓
LLM-Antwortgenerierung
    ↓
Juristische Antwort



========================================
⚖ LEGAL RAG SYSTEM ACTIVE
========================================
Ihre Frage (oder 'exit'): Darf der Vermieter die Miete nach Modernisierung erhöhen?

🔍 [Query Analyse] Suchphrase: 'Mieterhöhung nach Modernisierung' | Aktive Filter-§: []

========================================
🤖 RECHTSSICHERE ANTWORT:
========================================
Gemäß § 559 Abs. 1 BGB kann der Vermieter nach einer Modernisierungsmaßnahme die jährliche Miete um...

## datei_2_rag.py: without px 

## datei_2_rag.py: with px
Programm A <---- Socket ----> Netzwerk <---- Socket ----> Programm B

    Python
       |
       | socket()
       V
    +----------------+
    | Betriebssystem |
    +----------------+
             |
             V
       Socket erzeugt
   

# ⚖️ Automatische Proxy-, VPN- und SSL-Konfiguration

Dieser Code-Block regelt vollautomatisch die sichere Verbindung zur OpenAI-API – egal, ob das Skript im **Büro vor Ort**, im **Homeoffice mit VPN (und Px)** oder im **reinen Homeoffice (ohne Px)** gestartet wird.

---

### Der Konfigurations-Code

```python
if check_if_px_is_running():
    print("-> Px-Proxy aktiv. Teste SSL-Konfiguration...")
    proxy_url = "[http://127.0.0.1:3128](http://127.0.0.1:3128)"
    
    try:
        # -----------------------------------------------------------------
        # 1. VERSUCH: Büro-Modus (Physisch im Firmennetzwerk)
        # -----------------------------------------------------------------
        # Wir konfigurieren httpx mit dem lokalen Px-Proxy und dem 
        # firmeneigenen Root-Zertifikat für die SSL-Inspektion.
        custom_http_client = httpx.Client(proxy=proxy_url, verify="firmen_cert.pem")
        custom_http_async_client = httpx.AsyncClient(proxy=proxy_url, verify="firmen_cert.pem")
        
        # TECHNISCHER TEST-AUFRUF:
        # Ein Zertifikatsfehler (SSLError) fliegt in Python erst, wenn echte Daten 
        # gesendet werden. Wir zwingen httpx hier zu einem schnellen Check.
        custom_http_client.get("[https://api.openai.com](https://api.openai.com)")
        print("   -> [Büro-Modus] Firmen-Zertifikat erfolgreich verifiziert.")
        
    except Exception as e:
        # -----------------------------------------------------------------
        # 2. VERSUCH (FALLBACK): Homeoffice-Modus (Mit VPN & gestartetem Px)
        # -----------------------------------------------------------------
        # Wenn der Test oben abstürzt (knallt), liegt das daran, dass das VPN 
        # den Datenverkehr ungefiltert durchlässt (Keine SSL-Inspektion).
        # OpenAI sendet sein Original-Zertifikat, das nicht zur 'firmen_cert.pem' passt.
        print(f"   -> [HO-Modus] Firmen-Zertifikat passt nicht. Wechsel zu Standard-SSL (verify=True).")
        
        # Lösung: Wir nutzen weiterhin den Proxy, prüfen aber gegen die globalen Internet-Zertifikate.
        custom_http_client = httpx.Client(proxy=proxy_url, verify=True)
        custom_http_async_client = httpx.AsyncClient(proxy=proxy_url, verify=True)
else:
    # ---------------------------------------------------------------------
    # 3. WEG: Reines Homeoffice (Ohne Px-Tool)
    # ---------------------------------------------------------------------
    # Keine lokalen Tunnel aktiv. Direkte, sichere Verbindung über das 
    # normale Internet mit Standard-Zertifikatsprüfung.
    print("-> Kein Px gefunden. Verwende direkte Internetverbindung (Homeoffice ohne Px).")
    custom_http_client = httpx.Client(verify=True)
    custom_http_async_client = httpx.AsyncClient(verify=True)

In [ ]:
wie lang ist die Kündigungsfrist für mich wenn ich 4 Jahre in der Wohnung schon lebe?

# Version 1
update mit ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
wie lang ist die Kündigungsfrist für mich wenn ich 4 Jahre in der Wohnung schon lebe?

In [6]:
import os
os.environ["LANGCHAIN_OPENAI_TCP_KEEPALIVE"] = "0"
import socket
import urllib3
import ssl  # Neu importiert für moderne SSL-Handhabung
import httpx
from typing import List
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

# ============================================================
# AUTOMATISCHE PROXY- & VPN-ERKENNUNG (FÜR BÜRO UND HOMEOFFICE)
# ============================================================
def check_if_px_is_running():
    """Prüft, ob das Px-Tool auf dem lokalen Port 3128 aktiv ist."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1)  # Wartet maximal 1 Sekunde auf Antwort
        s.connect(("127.0.0.1", 3128))
        s.close()
        return True
    except (socket.timeout, ConnectionRefusedError, OSError):
        return False

# Vorbereitung: SSL-Prüfung modern deaktivieren (entspricht verify=False)
#ssl_context = ssl.create_default_context()
# Das ist die explizite, modernste Variante für HTTP-Clients:
ssl_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
ssl_context.check_hostname = False
ssl_context.verify_mode = ssl.CERT_NONE

# Initialisiere die Verbindungsparameter für LangChain
custom_http_client = None
custom_http_async_client = None

if check_if_px_is_running():
    print("-> Büro-Situation erkannt (oder Px ist aktiv): Schalte Px-Proxy ein.")
    os.environ["HTTP_PROXY"] = "http://127.0.0.1:3128"
    os.environ["HTTPS_PROXY"] = "http://127.0.0.1:3128"
    
    # SSL-Prüfung global deaktivieren und Warnungen unterdrücken
    os.environ["CURL_CA_BUNDLE"] = ""
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    # Moderne Transport-Zuweisung für HTTP und HTTPS via Px
    px_transport = httpx.HTTPTransport(proxy="http://127.0.0.1:3128", verify=ssl_context)
    
    custom_http_client = httpx.Client(
        mounts={"http://": px_transport, "https://": px_transport}
    )
    custom_http_async_client = httpx.AsyncClient(
        mounts={"http://": px_transport, "https://": px_transport}
    )
else:
    print("-> Homeoffice-Situation erkannt (Px ist inaktiv): Nutze Bosch-Firmen-Proxy.")

    proxy_url = "http://rb-proxy-de.bosch.com:8080"

    os.environ["HTTP_PROXY"] = proxy_url
    os.environ["HTTPS_PROXY"] = proxy_url

    # Ollama lokal niemals über Firmenproxy
    os.environ["NO_PROXY"] = "localhost,127.0.0.1"

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    # Moderne Transport-Zuweisung für Bosch-Proxy mit Timeout
    bosch_transport = httpx.HTTPTransport(proxy=proxy_url, verify=ssl_context)

    custom_http_client = httpx.Client(
        mounts={"http://": bosch_transport, "https://": bosch_transport},
        timeout=60.0
    )

    custom_http_async_client = httpx.AsyncClient(
        mounts={"http://": bosch_transport, "https://": bosch_transport},
        timeout=60.0
    )
# =========


    
#======================================================
# CONFIG & INITIALISIERUNG
# ============================================================
load_dotenv()
PERSIST_DIRECTORY = "chroma_legal_rag"

# Übergabe der passenden http_client-Parameter (Löst die UserWarnings auf)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    http_client=custom_http_client,
    http_async_client=custom_http_async_client
)

llm_query = ChatOpenAI(
    model="gpt-5.4-nano", 
    temperature=0.1,
    http_client=custom_http_client,
    http_async_client=custom_http_async_client
)

llm_answer = ChatOpenAI(
    model="gpt-5.4-mini", 
    temperature=0,
    http_client=custom_http_client,
    http_async_client=custom_http_async_client
)

vector_store = Chroma(
    persist_directory=PERSIST_DIRECTORY,
    embedding_function=embeddings
)

# ============================================================
# METADATEN-EXTRAKTION (QUERY ROUTING)
# ============================================================
class QueryExtraction(BaseModel):
    search_query: str = Field(description="Optimierte, suchbare juristische Kernphrase.")
    paragraph_filter: List[str] = Field(default_factory=list, description="Liste reiner Paragraphen-Nummern ohne Paragraphenzeichen.")

query_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Du bist ein präziser juristischer Such-Assistent. Extrahiere die suchbare Kernphrase.\n"
        "Erkennst du konkrete Paragraphen-Nennungen (z.B. '§ 556d' oder 'Paragraph 573'), extrahiere NUR die reine Nummer/Ziffer (z.B. '556d', '573') in die Liste paragraph_filter."
    ),
    ("human", "{question}")
])

query_chain = query_prompt | llm_query.with_structured_output(QueryExtraction)

# ============================================================
# RETRIEVAL STAGE (MIT METADATEN-FILTERUNG & FALLBACK)
# ============================================================
def retrieve_documents(input_data: dict):
    question = input_data["question"] if isinstance(input_data, dict) else input_data

    try:
        extracted = query_chain.invoke({"question": question})
        search_phrase = extracted.search_query if extracted.search_query else question
        p_filters = extracted.paragraph_filter if extracted.paragraph_filter else []
    except Exception as e:
        print(f"  ⚠ Query Extraction fehlgeschlagen: {e}")
        search_phrase = question
        p_filters = []

    print(f"\n🔍 [Query Analyse] Suchphrase: '{search_phrase}' | Aktive Filter-§: {p_filters}")

    search_kwargs = {"k": 5}
    has_filter = False

    if p_filters:
        clean_filters = [str(p).replace("§", "").strip() for p in p_filters if p]
        
        if clean_filters:
            has_filter = True
            if len(clean_filters) == 1:
                search_kwargs["filter"] = {"paragraph": clean_filters[0]}
            else:
                search_kwargs["filter"] = {"$or": [{"paragraph": p} for p in clean_filters]}

    retriever = vector_store.as_retriever(search_type="mmr", search_kwargs=search_kwargs)
    docs = retriever.invoke(search_phrase)

    if not docs and has_filter:
        print("  🔀 [Fallback] Keine Dokumente mit Metadaten-Filter gefunden. Starte ungefilterte Vektorsuche...")
        fallback_kwargs = {"k": 3}
        fallback_retriever = vector_store.as_retriever(search_type="mmr", search_kwargs=fallback_kwargs)
        docs = fallback_retriever.invoke(search_phrase)

    return {"question": question, "docs": docs}

# ============================================================
# GENERATION STAGE
# ============================================================
answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Beantworte die juristische Frage des Nutzers AUSSCHLIESSLICH basierend auf dem bereitgestellten Kontext. "
        "Wenn der Kontext die Antwort nicht hergibt, sage das sachlich. "
        "Gehe strukturiert vor, nenne immer die exakte Quelle (§, Absatz, Nummer) und bleibe rechtssicher.\n\n"
        "Kontext:\n{context}"
    ),
    ("human", "{question}")
])

def generate_answer(input_data: dict):
    if not isinstance(input_data, dict) or "docs" not in input_data:
        return "Fehler in der Retrieval-Verarbeitungskette."

    question = input_data["question"]
    docs = input_data["docs"]

    if not docs:
        return "Ich konnte keine passenden oder relevanten gesetzlichen Bestimmungen in der Datenbank finden."

    context_str = "\n\n".join([doc.page_content for doc in docs])
    
    prompt = answer_prompt.format(
        context=context_str,
        question=question
    )

    response = llm_answer.invoke(prompt)
    return response.content

# ============================================================
# PIPELINE STARTEN
# ============================================================
# ============================================================
# PIPELINE INITIALISIERUNG
# ============================================================
rag_chain = (
    {"question": RunnablePassthrough()}

    | RunnableLambda(retrieve_documents)
    | RunnableLambda(generate_answer)
)

# ============================================================
# INTERFACE
# ============================================================
if __name__ == "__main__":
    while True:
        print("\n" + "="*40)
        print("⚖ LEGAL RAG SYSTEM ACTIVE")
        print("="*40)

        user_input = input("Ihre Frage (oder 'exit'): ")
        if user_input.lower() in ["exit", "quit"]:
            print("System wird beendet. Auf Wiedersehen!")
            break
        if not user_input.strip():
            continue

        # Der String aus 'user_input' wird durch RunnablePassthrough() 
        # automatisch als Wert für den Key 'question' übergeben
        answer_output = rag_chain.invoke(user_input)

        print("\n" + "="*40)
        print("🤖 RECHTSSICHERE ANTWORT:")
        print("="*40)
        print(answer_output)

-> Homeoffice-Situation erkannt (Px ist inaktiv): Nutze Bosch-Firmen-Proxy.

⚖ LEGAL RAG SYSTEM ACTIVE


Ihre Frage (oder 'exit'):  wie lang ist die Kündigungsfrist für mich wenn ich 4 Jahre in der Wohnung schon lebe?



🔍 [Query Analyse] Suchphrase: 'Kündigungsfrist wie lang wenn Mieter 4 Jahre in der Wohnung lebt' | Aktive Filter-§: []

🤖 RECHTSSICHERE ANTWORT:
Nach dem bereitgestellten Kontext gilt für **Mieter** die **ordentliche Kündigungsfrist von drei Monaten**.

**Quelle:** § 573c Abs. 1 BGB  
- „Die Kündigung ist spätestens am dritten Werktag eines Kalendermonats zum Ablauf des übernächsten Monats zulässig.“

Für Ihre **4 Jahre Mietdauer** ergibt sich aus dem Kontext **noch keine Verlängerung** für den Mieter; die im Kontext genannte Verlängerung betrifft ausdrücklich **den Vermieter** nach **fünf und acht Jahren**.  
**Quelle:** § 573c Abs. 1 BGB  
- „Die Kündigungsfrist für den Vermieter verlängert sich nach fünf und acht Jahren seit der Überlassung des Wohnraums um jeweils drei Monate.“

Wenn Sie möchten, kann ich Ihnen auch sagen, **bis wann genau** Sie die Kündigung in einem konkreten Monat einreichen müssen.

⚖ LEGAL RAG SYSTEM ACTIVE


Ihre Frage (oder 'exit'):  exit


System wird beendet. Auf Wiedersehen!
